## 1. Load Raw Data

In [1]:
import pandas as pd

In [22]:
customers = pd.read_csv("../data/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/olist_orders_dataset.csv")
products = pd.read_csv("../data/olist_products_dataset.csv")
sellers = pd.read_csv("../data/olist_sellers_dataset.csv")
product_category_name_translation = pd.read_csv("../data/product_category_name_translation.csv")

## 2. Loaded DataFrames Inspection

In [10]:
def inspect_table(df, table_name):
    print("=====================================")
    print(f"TABLE: {table_name}")
    print("-------------------------------------")

    print("Shape:", df.shape)
    print("-------------------------------------")

    print("\nColumns:")
    print(df.columns.tolist())
    print("-------------------------------------")

    print("\nData Types:")
    print(df.dtypes)
    print("-------------------------------------")

    print("\nFirst 5 Rows:")
    print(df.head())
    print("-------------------------------------")

    print("\nMissing Values:")
    print(df.isnull().sum())
    print("-------------------------------------")
    
    print("\nDuplicated Values:")
    print(df.duplicated().sum())

    print("===================================== \n")

In [8]:
tables = {
    "Customers": customers,
    "Geolocation": geolocation,
    "Order Items": order_items,
    "Order Payments": order_payments,
    "Order Reviews": order_reviews,
    "Orders": orders,
    "Products": products,
    "Sellers": sellers,
    "Category Translation": product_category_name_translation
}

In [11]:
for table_name, df in tables.items():
    inspect_table(df, table_name)

TABLE: Customers
-------------------------------------
Shape: (99441, 5)
-------------------------------------

Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
-------------------------------------

Data Types:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object
-------------------------------------

First 5 Rows:
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_

## 3. Clean Individual Tables

### 3.1 Orders

In [48]:
# Convert date columns from string to datetime
date_cols = ['order_purchase_timestamp', 'order_approved_at', 
             'order_delivered_carrier_date', 'order_delivered_customer_date', 
             'order_estimated_delivery_date']

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

**Note:** `order_approved_at`, `order_delivered_carrier_date`, and `order_delivered_customer_date` 
contain nulls for orders that haven't reached that stage yet (e.g. `canceled`, `processing`, `shipped`). 
These nulls are kept as-is — they're not missing data, they're accurate: an order that was canceled 
should not have a delivery date. All order statuses are kept (not filtered to `delivered` only), so 
each analysis question can filter by status as needed.

In [14]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

### 3.2 Order Items

In [49]:
# Convert date column
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

# Check for invalid numeric values (nulls alone won't catch these)
print("Rows with price <= 0:", (order_items['price'] <= 0).sum())
print("Rows with freight_value < 0:", (order_items['freight_value'] < 0).sum())
print()
print(order_items[['price', 'freight_value']].describe())

Rows with price <= 0: 0
Rows with freight_value < 0: 0

               price  freight_value
count  112650.000000  112650.000000
mean      120.653739      19.990320
std       183.633928      15.806405
min         0.850000       0.000000
25%        39.900000      13.080000
50%        74.990000      16.260000
75%       134.900000      21.150000
max      6735.000000     409.680000


### 3.3 Customers

In [17]:
# Check for duplicate customer_unique_id 
# (a customer can place multiple orders, so customer_id is unique per order,
# but customer_unique_id identifies the actual person)
print("Unique customer_id:", customers['customer_id'].nunique())
print("Unique customer_unique_id:", customers['customer_unique_id'].nunique())
print()
print("customer_state value counts (top 10):")
print(customers['customer_state'].value_counts().head(10))

Unique customer_id: 99441
Unique customer_unique_id: 96096

customer_state value counts (top 10):
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64


### 3.4 Products

In [23]:
# Fill missing category with 'unknown' instead of dropping rows
products['product_category_name'] = products['product_category_name'].fillna('unknown')

# Check the 2 rows missing dimensions - inspect before deciding
products[products['product_weight_g'].isnull()]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# Re-run from the filled state (category nulls -> 'unknown')
# Drop only the one specific row that's almost entirely empty
products = products[products['product_id'] != '5eb564652db742ff8f28759cd8d2652a']

print("Shape:", products.shape)
print()
print("Nulls remaining:")
print(products.isnull().sum())

Shape: (32950, 9)

Nulls remaining:
product_id                      0
product_category_name           0
product_name_lenght           609
product_description_lenght    609
product_photos_qty            609
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64


### 3.5 Sellers

In [25]:
print("Unique seller_id:", sellers['seller_id'].nunique())
print("Total rows:", len(sellers))
print()
print("seller_state value counts (top 10):")
print(sellers['seller_state'].value_counts().head(10))

Unique seller_id: 3095
Total rows: 3095

seller_state value counts (top 10):
seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
Name: count, dtype: int64


### 3.6 Payments

In [27]:
print("Total rows:", len(order_payments))
print("Unique order_id:", order_payments['order_id'].nunique())
print()
print("payment_type value counts:")
print(order_payments['payment_type'].value_counts())
print()
print("payment_installments range:", order_payments['payment_installments'].min(), "to", order_payments['payment_installments'].max())

Total rows: 103886
Unique order_id: 99440

payment_type value counts:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

payment_installments range: 0 to 24


In [30]:
# Inspect rows with 0 installments
order_payments[order_payments['payment_installments'] == 0]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [31]:
# Inspect rows with payment_type = not_defined
order_payments[order_payments['payment_type'] == 'not_defined']

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [32]:
# Fix logically invalid installment count (0 -> 1, since a real payment occurred)
order_payments.loc[order_payments['payment_installments'] == 0, 'payment_installments'] = 1

# Verify fix
print("Rows with 0 installments now:", (order_payments['payment_installments'] == 0).sum())

Rows with 0 installments now: 0


### 3.7 Order Reviews

In [33]:
print("Total rows:", len(order_reviews))
print("Unique order_id:", order_reviews['order_id'].nunique())
print("Unique review_id:", order_reviews['review_id'].nunique())
print()
print("review_score value counts:")
print(order_reviews['review_score'].value_counts().sort_index())
print()
print("Duplicate order_id count:", order_reviews['order_id'].duplicated().sum())

Total rows: 99224
Unique order_id: 98673
Unique review_id: 98410

review_score value counts:
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

Duplicate order_id count: 551


In [34]:
# Keep only the most recent review per order_id
order_reviews_clean = order_reviews.sort_values('review_creation_date').drop_duplicates(
    subset='order_id', keep='last'
)

print("Rows before:", len(order_reviews))
print("Rows after:", len(order_reviews_clean))
print("Unique order_id after:", order_reviews_clean['order_id'].nunique())

Rows before: 99224
Rows after: 98673
Unique order_id after: 98673


### 3.8 Category Translation

In [35]:
# Check if every category in products actually has a translation available
categories_in_products = set(products['product_category_name'].unique())
categories_in_translation = set(product_category_name_translation['product_category_name'].unique())

missing_translations = categories_in_products - categories_in_translation
print("Categories in products but missing from translation table:", missing_translations)

Categories in products but missing from translation table: {'pc_gamer', 'unknown', 'portateis_cozinha_e_preparadores_de_alimentos'}


In [36]:
# Manually add missing category translations
missing_rows = pd.DataFrame({
    'product_category_name': ['unknown', 'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'],
    'product_category_name_english': ['unknown', 'pc_gamer', 'kitchen_portable_appliances']
})

product_category_name_translation = pd.concat(
    [product_category_name_translation, missing_rows], ignore_index=True
)

# Verify no more gaps
categories_in_translation = set(product_category_name_translation['product_category_name'].unique())
print("Still missing:", categories_in_products - categories_in_translation)

Still missing: set()


## 4. Merge Tables

Starting from `order_items` (the spine — one row per product per order), and merging outward.
Row counts are checked after every merge to catch unexpected duplication early.

In [51]:
# Merge product category English translation into products first,
# so 'products' is fully ready before joining it to the main spine
products_full = products.merge(
    product_category_name_translation, on='product_category_name', how='left'
)

print("products rows before:", len(products))
print("products_full rows after:", len(products_full))
print("Nulls in product_category_name_english:", products_full['product_category_name_english'].isnull().sum())

products rows before: 32950
products_full rows after: 32950
Nulls in product_category_name_english: 0


In [52]:
analysis_df = order_items.merge(orders, on='order_id', how='left')

print("order_items rows:", len(order_items))
print("analysis_df rows after merge:", len(analysis_df))

order_items rows: 112650
analysis_df rows after merge: 112650


In [53]:
analysis_df = analysis_df.merge(customers, on='customer_id', how='left')

print("Rows after merging customers:", len(analysis_df))

Rows after merging customers: 112650


In [54]:
analysis_df = analysis_df.merge(products_full, on='product_id', how='left')

print("Rows after merging products_full:", len(analysis_df))
print("Nulls in product_category_name_english:", analysis_df['product_category_name_english'].isnull().sum())

Rows after merging products_full: 112650
Nulls in product_category_name_english: 17


In [55]:
# Find which order_items rows have no matching product info after the merge
missing_products = analysis_df[analysis_df['product_category_name_english'].isnull()]
print("Rows with missing product info:", len(missing_products))
print()
print("Unique product_ids involved:", missing_products['product_id'].nunique())
print()
print(missing_products[['order_id', 'product_id']])

Rows with missing product info: 17

Unique product_ids involved: 1

                                order_id                        product_id
7098    101157d4fae1c9fb74a00a5dee265c25  5eb564652db742ff8f28759cd8d2652a
9233    1521c6bb7b1028154c8c67cf80fa809f  5eb564652db742ff8f28759cd8d2652a
28715   415cfaaaa8cea49f934470548797fed1  5eb564652db742ff8f28759cd8d2652a
28716   415cfaaaa8cea49f934470548797fed1  5eb564652db742ff8f28759cd8d2652a
39299   595316a07cd3dea9db7adfcc7e247ae7  5eb564652db742ff8f28759cd8d2652a
48424   6e150190fbe04c642a9cf0b80d83ee16  5eb564652db742ff8f28759cd8d2652a
48980   6f497c40431d5fb0cfbd6c943dd29215  5eb564652db742ff8f28759cd8d2652a
71134   a2456e7f02197951664897a94c87242d  5eb564652db742ff8f28759cd8d2652a
73556   a7a43f469c0d7bdb0a23a82db125aefa  5eb564652db742ff8f28759cd8d2652a
82767   bbfc7badbed2f1828e22b6d629201bd4  5eb564652db742ff8f28759cd8d2652a
84199   bf49f84a0580ef6751e13357776b7ed9  5eb564652db742ff8f28759cd8d2652a
85060   c1424efcde3c9e9febd9e176

**Note:** 17 rows have null `product_category_name_english` after this merge. All 17 trace back to a 
single `product_id` that was dropped in Section 3.4 due to having no usable product information. 
These rows are kept in `analysis_df` since they still contain valid order/customer/price data — 
they will simply be excluded from any category-specific groupby analysis.

In [56]:
# Aggregate payments to one row per order_id before merging
payments_agg = order_payments.groupby('order_id').agg(
    total_payment_value=('payment_value', 'sum'),
    payment_installments_max=('payment_installments', 'max'),
    payment_type_first=('payment_type', 'first')
).reset_index()

print("order_payments rows:", len(order_payments))
print("payments_agg rows:", len(payments_agg))

order_payments rows: 103886
payments_agg rows: 99440


In [57]:
analysis_df = analysis_df.merge(payments_agg, on='order_id', how='left')

print("Rows after merging payments_agg:", len(analysis_df))
print("Nulls in total_payment_value:", analysis_df['total_payment_value'].isnull().sum())

Rows after merging payments_agg: 112650
Nulls in total_payment_value: 3


In [58]:
missing_payments = analysis_df[analysis_df['total_payment_value'].isnull()]
print("Unique order_ids with missing payment:", missing_payments['order_id'].nunique())
print()
print(missing_payments[['order_id', 'order_status']])

Unique order_ids with missing payment: 1

                               order_id order_status
84389  bfbd0f9bdef84302105ad712db648a6c    delivered
84390  bfbd0f9bdef84302105ad712db648a6c    delivered
84391  bfbd0f9bdef84302105ad712db648a6c    delivered


**Note:** 1 order (`bfbd0f9bdef84302105ad712db648a6c`, status `delivered`) has no matching payment 
record in the source data, despite being delivered. This appears to be a genuine gap in the raw 
dataset rather than a business rule (unlike canceled orders lacking delivery dates). Left as NaN 
rather than imputed, since there's no reliable way to know the actual amount paid. Affects only 
this 1 order (3 rows in `analysis_df` due to multiple line items).

In [59]:
analysis_df = analysis_df.merge(
    order_reviews_clean[['order_id', 'review_score']], on='order_id', how='left'
)

print("Rows after merging reviews:", len(analysis_df))
print("Nulls in review_score:", analysis_df['review_score'].isnull().sum())

Rows after merging reviews: 112650
Nulls in review_score: 942


**Note:** 942 rows have no `review_score` — expected, since not every delivered order gets a 
customer review. Left as NaN; will be naturally excluded from review-based aggregations.

In [60]:
analysis_df = analysis_df.merge(sellers, on='seller_id', how='left')

print("Rows after merging sellers:", len(analysis_df))
print("Nulls in seller_state:", analysis_df['seller_state'].isnull().sum())

Rows after merging sellers: 112650
Nulls in seller_state: 0


## 5. Final Validation & Save

In [61]:
print("Final shape:", analysis_df.shape)
print()
print(analysis_df.dtypes)
print()
print("Total nulls per column:")
print(analysis_df.isnull().sum()[analysis_df.isnull().sum() > 0])

Final shape: (112650, 34)

order_id                                 object
order_item_id                             int64
product_id                               object
seller_id                                object
shipping_limit_date              datetime64[ns]
price                                   float64
freight_value                           float64
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
customer_unique_id                       object
customer_zip_code_prefix                  int64
customer_city                            object
customer_state                           object
product_category_name                    object
product_name_lenght                     float64
product_descr

In [62]:
import os
os.makedirs('../data', exist_ok=True)

analysis_df.to_csv('../data/analysis_df.csv', index=False)

print("Saved successfully.")
print("Final shape:", analysis_df.shape)

Saved successfully.
Final shape: (112650, 34)


## Summary

- Loaded 9 raw Olist CSV files
- Cleaned each table individually: fixed datatypes, handled nulls with reasoning per case 
  (filled where a row remained useful, dropped only where a row had no usable information, 
  left NaN where the value was genuinely unknown rather than fabricating it)
- Deduplicated `order_reviews` (551 orders had multiple reviews) and aggregated `order_payments` 
  (installments) to prevent row multiplication before merging
- Built `analysis_df` by merging outward from `order_items` (the spine), through 6 merges, 
  with row-count and null verification after each step
- Final shape: 112,650 rows × 34 columns
- Saved to `../data/analysis_df.csv` for use in subsequent EDA notebooks